In [1]:
import numpy as np
import glob
import os

In [2]:
# Configuration
data_dir = "/mnt/c/Users/obbee/research/notebooks/ML/npy_data"
hits_pattern = os.path.join(data_dir, "hits_batch_*.npy")
info_pattern = os.path.join(data_dir, "group_info_batch_*.npy")

hits_files = sorted(glob.glob(hits_pattern))
info_files = sorted(glob.glob(info_pattern))

print(f"Found {len(hits_files)} hits files and {len(info_files)} info files.")

if not hits_files:
    print("No files found! Please run the converter first.")

Found 145 hits files and 145 info files.


In [3]:
# Load the first batch
if hits_files:
    batch_idx = 0
    hits_file = hits_files[batch_idx]
    info_file = info_files[batch_idx]

    print(f"Loading batch {batch_idx}...")
    print(f"Hits: {hits_file}")
    print(f"Info: {info_file}")

    hits_data = np.load(hits_file, allow_pickle=True)
    group_info = np.load(info_file)

    print(f"Hits array shape: {hits_data.shape}")
    print(f"Group info shape: {group_info.shape}")

Loading batch 0...
Hits: /mnt/c/Users/obbee/research/notebooks/ML/npy_data/hits_batch_0.npy
Info: /mnt/c/Users/obbee/research/notebooks/ML/npy_data/group_info_batch_0.npy
Hits array shape: (11,)
Group info shape: (11, 18)


In [8]:
# Inspect a specific group
if hits_files:
    group_idx = 1

    print(f"\n--- Inspecting Group {group_idx} ---")

    # Group Info
    # [pionInGroup, muonInGroup, MIPinGroup, pionStopX, pionStopY, pionStopZ, 
    #  totalPionEnergy, totalMuonEnergy, totalMIPEnergy, theta, phi, eventID, 
    #  startX, startY, startZ, endX, endY, endZ]
    info = group_info[group_idx]
    print("Group Info:")
    print(f"  Event ID: {int(info[11])}")
    print(f"  Flags: Pion={int(info[0])}, Muon={int(info[1])}, MIP={int(info[2])}")
    print(f"  Pion Stop: ({info[3]:.2f}, {info[4]:.2f}, {info[5]:.2f})")
    print(f"  Energies: Pion={info[6]:.2f}, Muon={info[7]:.2f}, MIP={info[8]:.2f}")
    print(f"  Angle: Theta={info[9]:.4f}, Phi={info[10]:.4f}")
    
    if len(info) >= 18:
        print(f"  Start: ({info[12]:.2f}, {info[13]:.2f}, {info[14]:.2f})")
        print(f"  End:   ({info[15]:.2f}, {info[16]:.2f}, {info[17]:.2f})")
    else:
        print("  Start/End: Not found (old data format)")

    # Hits
    # [coord, z, stripType, energySmeared, pdg_binary]
    hits = hits_data[group_idx]
    print(f"\nHits ({len(hits)} hits):")
    print("  Idx | Coord  | Z      | Type | Energy | PDG Mask")
    print("  ----------------------------------------------")
    for i, hit in enumerate(hits):
        print(f"  {i:3d} | {hit[0]:6.2f} | {hit[1]:6.2f} | {int(hit[2])}    | {hit[3]:6.2f} | {int(hit[4]):5d}")


--- Inspecting Group 1 ---
Group Info:
  Event ID: 0
  Flags: Pion=0, Muon=1, MIP=0
  Pion Stop: (4.16, 6.10, 3.54)
  Energies: Pion=0.00, Muon=3.98, MIP=0.00
  Angle: Theta=1.1919, Phi=-1.2292
  Start: (4.19, 6.02, 3.59)
  End:   (4.32, 5.31, 3.82)

Hits (4 hits):
  Idx | Coord  | Z      | Type | Energy | PDG Mask
  ----------------------------------------------
    0 |   5.50 |   3.81 | 1    |   1.13 |     2
    1 |   5.30 |   3.81 | 1    |   1.30 |     2
    2 |   4.30 |   3.67 | 0    |   1.21 |     2
    3 |   6.10 |   3.53 | 1    |   0.33 |     2


In [ ]:
# Sanity Check
if hits_files:
    print("\n--- Sanity Check ---")
    has_pion_flag = bool(info[0])
    pion_hits_found = False
    for hit in hits:
        mask = int(hit[4])
        # Check if bit 0 (PION=1) is set. PION is 0b00001
        if mask & 1:
            pion_hits_found = True
            break

    print(f"Pion Flag: {has_pion_flag}")
    print(f"Pion Hits Found: {pion_hits_found}")

    if has_pion_flag == pion_hits_found:
        print("SUCCESS: Pion flag matches hit content.")
    else:
        print("WARNING: Pion flag does not match hit content!")